# Autoship Nudge Promo Incentive — Power Analysis (Autoship Adoption Rate, 2-Cell Design, Observed Allocation Rate)

**Experiment:** Autoship Nudge Promo Incentive Test · **Owner:** Sergio Oyola · **Primary metric analyzed here:** Autoship Adoption Rate · **Randomization unit:** `client_id` · **Allocation point:** post-First-Fix checkout, once keep rate is known, at the moment the client selects a Quick Fix date and clicks "Schedule a Quick Fix" (prior to Narvar handoff)

This notebook sizes a standard 2-cell A/B version of the experiment for its primary metric, **Autoship Adoption Rate**, using the experiment's own live allocation data for daily volume and a reach-gated historical baseline for the adoption rate.

## Population
Manual clients — i.e., not already enrolled in Autoship — who completed First Fix checkout with a **Buy 1+** keep rate (kept at least one item). Buy 0 clients always see the BAU Quick Fix experience with no Autoship nudge and are out of scope for this comparison, per the PRD. Three additional checks apply to the baseline population (Step 1): `'Never Active'` state prior to First Fix (excludes reactivated clients), no employees, no fraud.

## Design: 2-cell test, single comparison
Eligible clients are randomized into 2 cells at a 50/50 split:

| Cell | Experience | Offer |
|---|---|---|
| Control | BAU Quick Fix, no Autoship nudge | None |
| Treatment | Autoship nudge + promo billboard | 10% off next eligible Fix |

A single pairwise comparison is planned: **Treatment vs. Control**, measuring the combined effect of introducing the Autoship nudge together with the promo incentive, relative to today's BAU experience. Because only one comparison is planned against the family-wise error budget, no multiple-comparison correction is needed here — sizing uses the initial `alpha = 0.05` directly.

## One-sided test
A flat result and a negative result both lead to the same rollout decision — don't ship the nudge+promo experience — so there's no need to distinguish "no effect" from "harmful." This sizing is **one-sided**, powered only to detect a positive lift.

## Why volume comes from the live allocation log
Qualifying for this test isn't the same as being randomized — that happens later, at the "Schedule a Quick Fix" click. A client who qualifies but never reaches that click is never randomized, so historical eligibility overstates true daily volume. Every row in the experiment's own allocation log is, by construction, a client who was actually randomized, so this notebook reads volume directly from that log instead.

## Metric definition
**Autoship Adoption Rate** = share of eligible clients who show a fresh Autoship demand event within a 90-day window following their First Fix checkout.

A client's Autoship history is tracked in `curated.client_pulse_journal`, a daily journal (one row per day any tracked client attribute changes) carrying `last_autoship_demand_ts` — the timestamp of that client's most recent Autoship demand event as of that journal row. A client is counted as **adopted** if, scanning their full journal history, the *earliest* `last_autoship_demand_ts` value that is itself later than their First Fix checkout date falls within 90 days of that checkout.

- **Full journal history is scanned, not a single snapshot** — `last_autoship_demand_ts` resets on full cancellation, so only the earliest post-First-Fix value is robust to that.
- **The demand timestamp must be strictly *after* First Fix checkout** — excludes clients whose Autoship timestamp predates First Fix entirely (a pre-existing enrollment unrelated to this nudge).

**Caveat:** this is an opt-in-*adjacent* signal (subscription creation, not a literal click event) and can't confirm the post-First-Fix nudge specifically caused the adoption.

In [1]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor
from power import n_total_statsmodels

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

# Data parameters
COHORT_START = '2025-08-01'
STABLE_WINDOW_END = '2026-06-01'  # exclusive upper bound: months on/after this date accelerate sharply with no counterpart in the live experiment's own allocation rate
MATURATION_DAYS = 90  # days to wait for a client's Autoship demand event to resolve

# The experiment allocates through two plans: a QA plan for internal validation traffic,
# and the primary plan for real client allocation. Only the primary plan is used below.
PRIMARY_PLAN_ID = '12bbfac2-30c0-4e39-860b-77240c23bb8f'

# Design parameters (2-cell test, single comparison, no multiple-comparison correction)
ALPHA = 0.05  # single comparison: Treatment vs. Control, no Bonferroni adjustment needed
POWER = 0.80
TWO_SIDED = False  # one-sided: only a positive lift over BAU changes the rollout decision
N_ARMS = 2
SPLIT = 0.5  # Control and Treatment are equal-sized arms
MDE_GRID = [0.02, 0.03, 0.04, 0.05, 0.10, 0.12, 0.15]  # relative lift on Autoship Adoption Rate, Treatment vs. Control

## Step 1 — Autoship Adoption Rate baseline

Baseline population: Manual + Buy 1+ First Fix, `'Never Active'` prior state, no employees, no fraud, pooled across `COHORT_START` through `STABLE_WINDOW_END`. The rate is further gated to clients confirmed to have clicked "Schedule a Quick Fix" in `curated.product_tracking_events` — that click is the actual allocation trigger, so a qualifying client isn't necessarily a randomized one. This baseline is unaffected by the live volume question below — it estimates the *rate* at which a client adopts, not how many clients enter the test per day.

In [2]:
reach_query = f"""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '{COHORT_START}'
    GROUP BY client_id, shipment_id
),
first_fix_deduped AS (
    SELECT client_id, checkout_date, autoship_or_manual, n_items_kept
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
),
state_at_fix AS (
    SELECT f.client_id, j.client_state_detail,
           ROW_NUMBER() OVER (PARTITION BY f.client_id ORDER BY j.start_timestamp DESC) AS rn
    FROM first_fix_deduped f
    JOIN curated.checkout_based_client_state_journal j
      ON j.client_id = f.client_id
     AND j.start_timestamp <= CAST(f.checkout_date AS TIMESTAMP)
),
eligible AS (
    SELECT f.client_id, f.checkout_date
    FROM first_fix_deduped f
    JOIN curated.client c ON c.client_id = f.client_id
    JOIN state_at_fix s ON s.client_id = f.client_id AND s.rn = 1
    WHERE f.autoship_or_manual = 'manual'
      AND f.n_items_kept >= 1
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
      AND s.client_state_detail = 'Never Active'
      AND f.checkout_date >= DATE '{COHORT_START}'
      AND f.checkout_date <  DATE '{STABLE_WINDOW_END}'
),
reached AS (
    SELECT DISTINCT e.client_id
    FROM eligible e
    JOIN curated.product_tracking_events t
      ON t.client_id = e.client_id
     AND t.name = 'schedule_quick_fix_button'
     AND t.action_name = 'schedule_quick_fix'
     AND t.screen_view_name = 'post_checkout_promo'
     AND t.event_timestamp >= CAST(e.checkout_date AS TIMESTAMP)
    WHERE t.date_in_utc >= DATE '{COHORT_START}'
),
fresh_demand AS (
    SELECT e.client_id, MIN(p.last_autoship_demand_ts) AS first_fresh_demand_ts
    FROM eligible e
    JOIN curated.client_pulse_journal p
      ON p.client_id = e.client_id
     AND p.last_autoship_demand_ts > e.checkout_date
    GROUP BY e.client_id
)
SELECT
    COUNT(DISTINCT e.client_id) AS n_eligible,
    COUNT(DISTINCT r.client_id) AS n_reached,
    COUNT(DISTINCT CASE WHEN r.client_id IS NOT NULL AND f.first_fresh_demand_ts IS NOT NULL
                         AND f.first_fresh_demand_ts <= e.checkout_date + INTERVAL '{MATURATION_DAYS}' DAY
                        THEN e.client_id END) AS n_adopted_reached
FROM eligible e
LEFT JOIN reached r ON r.client_id = e.client_id
LEFT JOIN fresh_demand f ON f.client_id = e.client_id
"""

reach_df = query(reach_query)
REACH_RATE = float(reach_df['n_reached'][0]) / float(reach_df['n_eligible'][0])
BASELINE_RATE = float(reach_df['n_adopted_reached'][0]) / float(reach_df['n_reached'][0])

print(f"REACH_RATE = {REACH_RATE:.1%}  |  BASELINE_RATE = {BASELINE_RATE:.4f}")
reach_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


REACH_RATE = 30.6%  |  BASELINE_RATE = 0.3361


,n_eligible,n_reached,n_adopted_reached
0,86892,26599,8940


## Step 2 — Observed daily allocation rate from the live experiment

The experiment's allocation log records one row per client at the moment they are actually randomized. This step measures the daily rate directly from that log, over the full period the experiment has been running so far.

In [3]:
allocation_query = f"""--sql
SELECT
    MIN(event_ts) AS first_allocation_ts,
    CURRENT_TIMESTAMP AS observation_ts,
    DATE_DIFF('hour', MIN(event_ts), CURRENT_TIMESTAMP) AS hours_observed,
    COUNT(DISTINCT rand_unit_value) AS n_allocated_total,
    COUNT(DISTINCT CASE WHEN cell_id = 1 THEN rand_unit_value END) AS n_control,
    COUNT(DISTINCT CASE WHEN cell_id = 2 THEN rand_unit_value END) AS n_treatment
FROM ab.current_allocations
WHERE plan_id = '{PRIMARY_PLAN_ID}'
"""

allocation_df = query(allocation_query)
allocation_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,first_allocation_ts,observation_ts,hours_observed,n_allocated_total,n_control,n_treatment
0,2026-08-12 20:57:06.866,2026-09-08 20:56:48.237 UTC,647,1992,1024,968


In [4]:
days_observed = float(allocation_df['hours_observed'][0]) / 24.0
DAILY_ELIGIBLE = float(allocation_df['n_allocated_total'][0]) / days_observed

print(f"days_observed = {days_observed:.2f}  |  DAILY_ELIGIBLE (observed) = {DAILY_ELIGIBLE:.1f} clients/day")

days_observed = 26.96  |  DAILY_ELIGIBLE (observed) = 73.9 clients/day


**Reading this:** this rate reflects the current run rate, not a projection — refresh as more allocation history accumulates, especially if the rollout is still ramping.

## Step 3 — Sample size & duration

`n_total_statsmodels` sizes a single pairwise 50/50 comparison (Treatment vs. Control); `n_treatment` is read as the **per-arm** requirement. Each of the 2 arms accrues `DAILY_ELIGIBLE / 2` clients per day under the 50/50 split, so `days_required = n_per_arm / (DAILY_ELIGIBLE / 2)`.

In [5]:
def size_table(rel_grid, baseline, daily):
    raw = n_total_statsmodels(
        baseline_rate=baseline, mde_relative=rel_grid, split_ratio=[SPLIT],
        alpha=ALPHA, power=POWER, two_sided=TWO_SIDED,
    )
    df = pd.DataFrame(raw).T.reset_index(drop=True)
    df['rel_effect'] = df['mde_relative'].apply(lambda x: f"{x:+.0%}")
    df['n_per_arm'] = df['n_treatment'].astype(int)
    df['n_total_2arm'] = df['n_per_arm'] * N_ARMS
    df['days_required'] = np.ceil(df['n_per_arm'] / (daily / N_ARMS)).astype(int)
    df['weeks_required'] = (df['days_required'] / 7).round(1)
    return df[['rel_effect', 'p_treatment', 'n_per_arm', 'n_total_2arm', 'days_required', 'weeks_required']]

sided = 'one-sided' if not TWO_SIDED else 'two-sided'
print(f"--- Autoship Adoption Rate, Treatment vs. Control (baseline={BASELINE_RATE:.1%}, alpha={ALPHA}, power={POWER:.0%}, {sided}, 50/50 split) ---")
size_table(MDE_GRID, BASELINE_RATE, DAILY_ELIGIBLE)

--- Autoship Adoption Rate, Treatment vs. Control (baseline=33.6%, alpha=0.05, power=80%, one-sided, 50/50 split) ---


,rel_effect,p_treatment,n_per_arm,n_total_2arm,days_required,weeks_required
0,+2%,0.342825,61359,122718,1661,237.3
1,+3%,0.346186,27336,54672,740,105.7
2,+4%,0.349547,15412,30824,418,59.7
3,+5%,0.352908,9887,19774,268,38.3
4,+10%,0.369713,2499,4998,68,9.7
5,+12%,0.376435,1743,3486,48,6.9
6,+15%,0.386518,1122,2244,31,4.4


**Reading this:** required duration is long because the live allocation rate is a small fraction of the broader qualifying population's volume — real drop-off between qualifying and reaching the allocation trigger. No harm/guardrail grid here: this sizes a one-sided positive MDE only; margin risk is covered qualitatively elsewhere (Experiment Design doc).

## Step 4 — Summary for the Experiment Design doc

Headline MDE below is a **placeholder 12% relative lift** on Autoship Adoption Rate (Treatment vs. Control) — the second-largest value in the Step 3 grid.

In [6]:
TARGET_REL_MDE = 0.12  # placeholder: 12% relative lift on Autoship Adoption Rate, Treatment vs. Control (second-largest value in the Step 3 grid)

res = n_total_statsmodels(
    baseline_rate=BASELINE_RATE,
    mde_relative=[TARGET_REL_MDE],
    split_ratio=[SPLIT],
    alpha=ALPHA,
    power=POWER,
    two_sided=TWO_SIDED,
)

n_per_arm = int(list(res.values())[0]['n_treatment'])
duration_days = int(np.ceil(n_per_arm / (DAILY_ELIGIBLE / N_ARMS)))

summary = {
    'Metric Used': 'Autoship Adoption Rate (Treatment vs. Control)',
    'Population': 'Manual clients, First Fix checkout complete, Buy 1+ keep rate, not already enrolled in Autoship, verified Never Active prior to First Fix, excluding employees and fraudulent clients, confirmed reached the click',
    'Baseline Value': f"{BASELINE_RATE:.1%} (pooled {COHORT_START} to {STABLE_WINDOW_END}, {MATURATION_DAYS}-day matured, among clients confirmed to have reached the click)",
    'Daily Eligible Volume': f"{DAILY_ELIGIBLE:,.1f} / day (observed live allocation rate, primary plan, {days_observed:.1f} days observed)",
    'Minimum Detectable Effect': f"+{TARGET_REL_MDE:.0%} relative ({BASELINE_RATE:.3f} -> {BASELINE_RATE*(1+TARGET_REL_MDE):.3f})",
    'One/Two-Sided Test': 'One-sided',
    'Significance Level': f"{ALPHA} (single comparison, no multiple-comparison correction)",
    'Statistical Power': f"{POWER:.0%}",
    'Variant Split %': '50% / 50% (Control / Treatment)',
    'Minimum Samples by Variant': f"{n_per_arm:,}",
    'Minimum Samples total (2 arms)': f"{n_per_arm*N_ARMS:,}",
    'Shortest Duration Required': f"{duration_days} days (~{duration_days/7:.1f} weeks)",
}
pd.Series(summary).to_frame('value')

,value
Metric Used,Autoship Adoption Rate (Treatment vs. Control)
Population,"Manual clients, First Fix checkout complete, Buy 1+ keep rate, not already enrolled in Autoship, verified Never Active prior to First Fix, excluding employees and fraudulent clients, confirmed reached the click"
Baseline Value,"33.6% (pooled 2025-08-01 to 2026-06-01, 90-day matured, among clients confirmed to have reached the click)"
Daily Eligible Volume,"73.9 / day (observed live allocation rate, primary plan, 27.0 days observed)"
Minimum Detectable Effect,+12% relative (0.336 -> 0.376)
One/Two-Sided Test,One-sided
Significance Level,"0.05 (single comparison, no multiple-comparison correction)"
Statistical Power,80%
Variant Split %,50% / 50% (Control / Treatment)
Minimum Samples by Variant,"1,743"


## Bottom line

- **Baseline is reach-gated and pooled** — measured only among clients confirmed to have clicked "Schedule a Quick Fix" (Step 1), across a stable multi-month historical window, not a single recent month.
- **Volume comes from the live allocation log**, not historical eligibility — a client only appears there once actually randomized, so this figure is measured directly, not modeled.
- **Required duration is long** because the live allocation rate is a small fraction of the broader qualifying population's volume — refresh as more allocation history accumulates.
- At **12% relative lift** (placeholder MDE), sample size and duration are in Step 4.